# 01 — Ingest Bronze Layer

Download stock price data with **yfinance**, load the CSV files, attach ingestion metadata, inspect the result, and save to the bronze layer.

**Bronze columns:** `date`, `ticker`, `open`, `high`, `low`, `close`, `adj_close`, `volume`, `source_file`, `loaded_at`

**Flow:** yfinance download → CSV → `load_stock_csv` → `add_ingestion_metadata` → `save_bronze_data`

## Setup

Add `src` to the Python path and import the bronze ingestion helpers.

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("..").resolve()
SRC_DIR = PROJECT_ROOT / "src"
RAW_DATA_DIR = PROJECT_ROOT / "data" / "sample"
BRONZE_OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "bronze" / "bronze_stock_prices.csv"

# Download settings
TICKERS = ["AAPL", "MSFT"]
START_DATE = "2024-01-01"
END_DATE = "2024-06-01"

sys.path.insert(0, str(SRC_DIR))

from ingestion import add_ingestion_metadata, download_stock_csv, load_stock_csv, save_bronze_data

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data dir: {RAW_DATA_DIR}")
print(f"Bronze output: {BRONZE_OUTPUT_PATH}")

Project root: C:\Users\bruce\OneDrive\Desktop\School\Personal Projects\financial-market-analytics-warehouse
Raw data dir: C:\Users\bruce\OneDrive\Desktop\School\Personal Projects\financial-market-analytics-warehouse\data\sample
Bronze output: C:\Users\bruce\OneDrive\Desktop\School\Personal Projects\financial-market-analytics-warehouse\data\processed\bronze\bronze_stock_prices.csv


## Download data with yfinance

Pull historical prices from Yahoo Finance and save one CSV per ticker to `data/sample/`.

Change `TICKERS`, `START_DATE`, and `END_DATE` in the setup cell above. If you omit dates, `download_stock_csv` uses `period="1y"` instead.

In [2]:
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

for ticker in TICKERS:
    output_file = RAW_DATA_DIR / f"{ticker}.csv"
    download_stock_csv(
        ticker=ticker,
        output_path=output_file,
        start_date=START_DATE,
        end_date=END_DATE,
    )
    print(f"Downloaded {ticker} -> {output_file}")

Downloaded AAPL -> C:\Users\bruce\OneDrive\Desktop\School\Personal Projects\financial-market-analytics-warehouse\data\sample\AAPL.csv
Downloaded MSFT -> C:\Users\bruce\OneDrive\Desktop\School\Personal Projects\financial-market-analytics-warehouse\data\sample\MSFT.csv


## Load raw CSV files

Reads every CSV in `data/sample/`. The ticker comes from the filename (e.g. `AAPL.csv` → `AAPL`).

In [3]:
csv_files = sorted(RAW_DATA_DIR.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(
        f"No CSV files found in {RAW_DATA_DIR}. Add stock CSV files and re-run this cell."
    )

bronze_frames = []
for csv_file in csv_files:
    raw_df = load_stock_csv(csv_file)
    bronze_frames.append(add_ingestion_metadata(raw_df, csv_file))

bronze_df = pd.concat(bronze_frames, ignore_index=True)

print(f"Loaded {len(csv_files)} file(s): {[f.name for f in csv_files]}")
print(f"Bronze rows: {len(bronze_df):,}")

Loaded 2 file(s): ['AAPL.csv', 'MSFT.csv']
Bronze rows: 210


## Inspect bronze data

Preview the combined dataset before saving.

In [4]:
bronze_df.head()

,Date,Open,High,Low,Close,Adj Close,Volume,source_file,ticker,loaded_at
0,2024-01-02,187.149994,188.440002,183.889999,185.639999,183.562164,82488700,AAPL.csv,AAPL,2026-06-22 21:05:32.981723+00:00
1,2024-01-03,184.220001,185.880005,183.429993,184.250000,182.187744,58414500,AAPL.csv,AAPL,2026-06-22 21:05:32.981723+00:00
2,2024-01-04,182.149994,183.089996,180.880005,181.910004,179.873932,71983600,AAPL.csv,AAPL,2026-06-22 21:05:32.981723+00:00
3,2024-01-05,181.990005,182.759995,180.169998,181.179993,179.152100,62379700,AAPL.csv,AAPL,2026-06-22 21:05:32.981723+00:00
4,2024-01-08,182.089996,185.600006,181.500000,185.559998,183.483063,59144500,AAPL.csv,AAPL,2026-06-22 21:05:32.981723+00:00


In [5]:
bronze_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 210 entries, 0 to 209
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype              
---  ------       --------------  -----              
 0   Date         210 non-null    object             
 1   Open         210 non-null    float64            
 2   High         210 non-null    float64            
 3   Low          210 non-null    float64            
 4   Close        210 non-null    float64            
 5   Adj Close    210 non-null    float64            
 6   Volume       210 non-null    int64              
 7   source_file  210 non-null    object             
 8   ticker       210 non-null    object             
 9   loaded_at    210 non-null    datetime64[us, UTC]
dtypes: datetime64[us, UTC](1), float64(5), int64(1), object(3)
memory usage: 16.5+ KB


## Save bronze layer

Write the ingested data to `data/processed/bronze/`.

In [6]:
save_bronze_data(bronze_df, BRONZE_OUTPUT_PATH)
print(f"Saved bronze data to {BRONZE_OUTPUT_PATH}")

Saved bronze data to C:\Users\bruce\OneDrive\Desktop\School\Personal Projects\financial-market-analytics-warehouse\data\processed\bronze\bronze_stock_prices.csv
